### Read SNOWPACK Output at TRAM — WY2026

Compare **HRRR-AK** and **HRRR-AK with lapse-rate corrected temperature** for
WY2026 only (October 1, 2025–September 30, 2026).

created by Cassie Lumbrazo\
updated: November 2026\
run location: UAS linux\
python environment: **xarray** (also runs in **rasterio**)

Reads the two `_base.smet` output files. Figures are displayed inline; no files
are exported and model outputs are not changed. This does not rerun SNOWPACK.
Observed snow depth comes from `JTMA2.2026-09-30.csv` and is compared with both
simulations. Blue = lapse-rate forcing, purple = raw forcing, dashed black =
TRAM observations. The CSV snow-depth units are converted from mm to cm.
The WY2026 forcing lacks the final two initialization days; inspect the actual
output coverage below rather than assuming a complete water year.

**Current output coverage:** both files contain 7,291 hourly records from
October 1, 2025 at 06:00 UTC through August 1, 2026 at 00:00 UTC. This is shorter
than the available forcing (through September 29). Plots show the full WY2026
calendar window; the blank tail is not filled or interpreted as zero snow.


In [ ]:
%matplotlib inline
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from IPython.display import display

sns.set_theme()
plt.rcParams['figure.figsize'] = [12, 6]


## Read the model outputs

Use the SMET header's `units_multiplier` and `units_offset` to convert the ASCII
values to the reported physical units. For example, stored snow depth is in cm
and is converted to m by the header's 0.01 multiplier; air temperature is stored
in °C and converted to K by its offset. Both raw and converted nodata sentinels
are masked. Plot snow depth in cm and SWE in mm water equivalent.


In [ ]:
def read_smet(filepath):
    """Read SNOWPACK ASCII output and apply its per-field unit metadata."""
    header = {}
    with Path(filepath).open() as f:
        for line_number, line in enumerate(f):
            line = line.strip()
            if line == '[DATA]':
                data_start = line_number + 1
                break
            if '=' in line and not line.startswith('#'):
                key, value = line.split('=', 1)
                header[key.strip()] = value.strip()
        else:
            raise ValueError(f'No [DATA] section in {filepath}')
    fields = header['fields'].split()
    if fields[0] != 'timestamp' or len(fields) != len(set(fields)):
        raise ValueError('Expected unique fields beginning with timestamp')
    frame = pd.read_csv(filepath, skiprows=data_start, sep=r'\s+',
                        names=fields, parse_dates=['timestamp']).set_index('timestamp')
    if frame.empty or frame.index.hasnans:
        raise ValueError(f'Empty output or invalid timestamps: {filepath}')
    if not frame.index.is_unique or not frame.index.is_monotonic_increasing:
        raise ValueError(f'Duplicate or unsorted output times: {filepath}')
    nodata = float(header.get('nodata', -999))
    frame = frame.replace(nodata, np.nan)
    multiplier = np.array(header.get('units_multiplier', ' '.join(['1'] * len(fields))).split(), dtype=float)
    offset = np.array(header.get('units_offset', ' '.join(['0'] * len(fields))).split(), dtype=float)
    units = header.get('plot_unit', ' '.join(['-'] * len(fields))).split()
    if not len(multiplier) == len(offset) == len(units) == len(fields):
        raise ValueError('Unit metadata does not match the field list')
    for i, name in enumerate(fields[1:], 1):
        frame[name] = frame[name] * multiplier[i] + offset[i]
        frame[name] = frame[name].mask(np.isclose(frame[name], nodata, rtol=0, atol=1e-8))
    ds = xr.Dataset.from_dataframe(frame)
    ds.attrs.update(header)
    for i, name in enumerate(fields[1:], 1):
        ds[name].attrs['units'] = units[i]
    return ds, header


In [ ]:
OUTPUT_DIR = Path('/home/cassie/python/models/run_snowpack/sites/tram/output')
WY_START = '2025-10-01T00:00:00'
WY_END = '2026-09-30T23:59:59'

paths = {
    'HRRR-AK lapse-rate forcing': OUTPUT_DIR / 'hrrrak_lapserate_tram_WY2026_base.smet',
    'HRRR-AK raw forcing': OUTPUT_DIR / 'hrrrak_tram_WY2026_base.smet',
}
colors = {'HRRR-AK lapse-rate forcing': 'tab:blue', 'HRRR-AK raw forcing': 'tab:purple'}
models, headers = {}, {}
for label, path in paths.items():
    ds, header = read_smet(path)
    ds = ds.sel(timestamp=slice(WY_START, WY_END))
    if ds.sizes['timestamp'] == 0:
        raise ValueError(f'No WY2026 output in {path}')
    if ds.HS_mod.attrs['units'] != 'm' or ds.SWE.attrs['units'] != 'kg/m2':
        raise ValueError(f'Unexpected snow-depth or SWE units in {path}')
    models[label], headers[label] = ds, header
    print(f"{label}: {ds.sizes['timestamp']:,} records; "
          f"{pd.Timestamp(ds.timestamp.values[0])} to "
          f"{pd.Timestamp(ds.timestamp.values[-1])} UTC")

ds_snowpack_hrrrak = models['HRRR-AK raw forcing']
ds_snowpack_hrrrak_lapserate = models['HRRR-AK lapse-rate forcing']
# Fail visibly if the runs cover different times rather than silently aligning.
xr.align(ds_snowpack_hrrrak, ds_snowpack_hrrrak_lapserate, join='exact')


In [ ]:
# Browse all available output variables and their converted units.
ds_snowpack_hrrrak


## Load TRAM snow-depth observations

Read the Synoptic station CSV, skip the units row by parsing timestamps, and
convert the declared **Millimeters** to **cm**. Timestamps are UTC, stored without
a timezone to match the model outputs. Hourly means combine duplicate timestamps;
missing hours remain NaN so lines break at gaps. The file's values are otherwise
plotted as downloaded: no offset adjustment or range filter is applied.
Observations continue beyond the August 1 model cutoff; simulations are not
extrapolated. Snow-depth observations are not used as SWE observations.


In [ ]:
OBS_FILE = Path('/hdd/snow_hydrology/met_station/tram/JTMA2.2026-09-30.csv')
observations_csv = pd.read_csv(OBS_FILE, comment='#', low_memory=False)
if observations_csv.loc[0, 'snow_depth_set_1'] != 'Millimeters':
    raise ValueError('Check the CSV snow-depth units before converting to cm')
observations_csv['time'] = pd.to_datetime(
    observations_csv['Date_Time'], utc=True, errors='coerce'
).dt.tz_convert(None)
# The units row has no timestamp and is excluded here.
observations = observations_csv.dropna(subset=['time']).set_index('time').sort_index()
if not observations['Station_ID'].eq('JTMA2').all():
    raise ValueError('Unexpected station in the TRAM observation CSV')
observations = observations.loc[WY_START:WY_END]
observations['hs'] = pd.to_numeric(observations['snow_depth_set_1'], errors='coerce') * 0.1
if observations.empty or not observations['hs'].notna().any():
    raise ValueError('No WY2026 snow-depth observations in the CSV')
duplicate_times = int(observations.index.duplicated().sum())
observed_hs = observations['hs'].resample('1h').mean()
ds_obs = observed_hs.to_xarray().to_dataset(name='hs')
ds_obs.hs.attrs.update(units='cm', long_name='Observed snow depth')
ds_obs.attrs.update(source=str(OBS_FILE), station_id='JTMA2', timezone='UTC')
print(f"TRAM observations: {int(ds_obs.hs.count()):,} valid hourly snow-depth values; "
      f"{pd.Timestamp(ds_obs.time.values[0])} to {pd.Timestamp(ds_obs.time.values[-1])} UTC")
print(f"Duplicate timestamps combined: {duplicate_times}; "
      f"missing hourly values retained: {int(ds_obs.hs.isnull().sum())}")
ds_obs


## Snow depth


In [ ]:
def format_axis(ax):
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b\n%Y'))
    ax.set_xlim(pd.Timestamp(WY_START), pd.Timestamp(WY_END))
    ax.set_xlabel('')
    ax.grid(alpha=0.3)

fig, ax = plt.subplots(figsize=(12, 6))
for label, ds in models.items():
    ax.plot(ds.timestamp, ds.HS_mod * 100, label=label,
            color=colors[label], linewidth=2.5)
ax.plot(ds_obs.time, ds_obs.hs, label='TRAM observations',
        color='black', linestyle='--', linewidth=2.0, zorder=5)
ax.set_title('SNOWPACK Simulations and Observed Snow Depth at TRAM — WY2026')
ax.set_ylabel('snow depth (cm)')
ax.set_ylim(bottom=0)
ax.legend(loc='upper left')
format_axis(ax)
fig.tight_layout()
plt.show()


### Alternative: observations as black points

Same data, model colors, figure size, and axes as above; observations use
black markers without connecting lines. Adjust `markersize` to compare density.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for label, ds in models.items():
    ax.plot(ds.timestamp, ds.HS_mod * 100, label=label,
            color=colors[label], linewidth=2.5)
ax.plot(ds_obs.time, ds_obs.hs, label='TRAM observations',
        color='black', linestyle='None', marker='o', markersize=3,
        markeredgewidth=0, zorder=5)
ax.set_title('SNOWPACK Simulations and Observed Snow Depth at TRAM — WY2026')
ax.set_ylabel('snow depth (cm)')
ax.set_ylim(bottom=0)
ax.legend(loc='upper left')
format_axis(ax)
fig.tight_layout()
plt.show()


### Presentation: HRRR-AK forcing and observed snow depth

Show the lapse-rate corrected simulation as **HRRR-AK forcing** in blue, with
black observation markers. This label is for presentation; the underlying model
remains the lapse-rate corrected run. Use a row legend configured for three
columns and end the displayed period at August 1, 2026. No figures are exported.


In [ ]:
PRESENTATION_END = pd.Timestamp('2026-08-01T00:00:00')
presentation_model = ds_snowpack_hrrrak_lapserate.sel(
    timestamp=slice(WY_START, PRESENTATION_END)
)
presentation_obs = ds_obs.sel(time=slice(WY_START, PRESENTATION_END))

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(presentation_model.timestamp, presentation_model.HS_mod * 100,
        label='HRRR-AK forcing', color='tab:blue', linewidth=2.5)
ax.plot(presentation_obs.time, presentation_obs.hs, label='TRAM observations',
        color='black', linestyle='None', marker='o', markersize=3,
        markeredgewidth=0, zorder=5)
ax.set_title('SNOWPACK Simulations and Observed Snow Depth at TRAM — WY2026')
ax.set_ylabel('snow depth (cm)')
ax.set_ylim(bottom=0)
ax.legend(loc='upper center', ncol=3, frameon=True)
format_axis(ax)
ax.set_xlim(pd.Timestamp(WY_START), PRESENTATION_END)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b\n%Y'))
fig.tight_layout()
plt.show()


### Presentation with semi-cleaned snow-depth observations

Apply a rough screening to a separate copy of the hourly observations:

- Mask values **above 200 cm before January 1, 2026**.
- Mask values **above 300 cm from January 1, 2026 onward**.
- Mask **zero values from December 1, 2025 through April 30, 2026**
  (December 1 inclusive, May 1 exclusive).

Rejected values become NaN without interpolation. The original observations,
CSV, and simulations are retained. Replot using the same blue model line and
black markers, ending August 1, with no x-axis title.


In [ ]:
# Rough observation screening; preserve the downloaded and original hourly data.
observation_time = ds_obs.time
before_january = observation_time < np.datetime64('2026-01-01')
large_before_january = before_january & (ds_obs.hs > 200)
large_from_january = (~before_january) & (ds_obs.hs > 300)
winter_zero = (
    (observation_time >= np.datetime64('2025-12-01'))
    & (observation_time < np.datetime64('2026-05-01'))
    & (ds_obs.hs == 0)
)
observation_rejected = large_before_january | large_from_january | winter_zero

ds_obs_semicleaned = ds_obs.copy(deep=True)
ds_obs_semicleaned['hs'] = ds_obs.hs.where(~observation_rejected)
ds_obs_semicleaned.hs.attrs.update(ds_obs.hs.attrs)
ds_obs_semicleaned.hs.attrs['screening'] = (
    'Mask >200 cm before 2026-01-01; >300 cm from 2026-01-01; '
    'zero values from 2025-12-01 inclusive to 2026-05-01 exclusive'
)
presentation_obs_semicleaned = ds_obs_semicleaned.sel(
    time=slice(WY_START, PRESENTATION_END)
)
print(f'Removed >200 cm before January 1: {int(large_before_january.sum())}')
print(f'Removed >300 cm from January 1: {int(large_from_january.sum())}')
print(f'Removed December–April zero values: {int(winter_zero.sum())}')
print(f'Valid hourly observations retained: {int(ds_obs_semicleaned.hs.count()):,}')


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(presentation_model.timestamp, presentation_model.HS_mod * 100,
        label='HRRR-AK forcing', color='tab:blue', linewidth=2.5)
ax.plot(presentation_obs_semicleaned.time, presentation_obs_semicleaned.hs, label='TRAM observations',
        color='black', linestyle='None', marker='o', markersize=3,
        markeredgewidth=0, zorder=5)
ax.set_title('SNOWPACK Simulations and Observed Snow Depth at TRAM — WY2026')
ax.set_ylabel('snow depth (cm)')
ax.set_ylim(-10, 400)  # Snow-depth range in cm; x-axis remains dates.
ax.legend(loc='upper center', ncol=3, frameon=True)
format_axis(ax)
ax.set_xlim(pd.Timestamp(WY_START), PRESENTATION_END)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b\n%Y'))
ax.set_xlabel('')
fig.tight_layout()
plt.show()


### Same figure with January 9, 2026 marked in red


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(presentation_model.timestamp, presentation_model.HS_mod * 100,
        label='HRRR-AK forcing', color='tab:blue', linewidth=2.5)
ax.plot(presentation_obs_semicleaned.time, presentation_obs_semicleaned.hs, label='TRAM observations',
        color='black', linestyle='None', marker='o', markersize=3,
        markeredgewidth=0, zorder=5)
ax.set_title('SNOWPACK Simulations and Observed Snow Depth at TRAM — WY2026')
ax.set_ylabel('snow depth (cm)')
ax.set_ylim(-10, 400)  # Snow-depth range in cm; x-axis remains dates.
ax.legend(loc='upper center', ncol=3, frameon=True)
format_axis(ax)
ax.set_xlim(pd.Timestamp(WY_START), PRESENTATION_END)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b\n%Y'))
ax.set_xlabel('')
ax.axvline(pd.Timestamp('2026-01-09'), color='red', linewidth=1.5,
           label='_nolegend_')
fig.tight_layout()
plt.show()


## Snow water equivalent


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for label, ds in models.items():
    # 1 kg/m² of SWE equals 1 mm of water equivalent.
    ax.plot(ds.timestamp, ds.SWE, label=label,
            color=colors[label], linewidth=2.5)
ax.set_title('SNOWPACK SWE at TRAM — WY2026')
ax.set_ylabel('SWE (mm water equivalent)')
ax.set_ylim(bottom=0)
ax.legend(loc='upper left')
format_axis(ax)
fig.tight_layout()
plt.show()


## Effect of the lapse-rate correction

Differences are **lapse-rate corrected minus uncorrected HRRR-AK**. These show
the model response to the temperature adjustment, not error against observations.


In [ ]:
difference = xr.Dataset({
    'snow_depth_cm': (ds_snowpack_hrrrak_lapserate.HS_mod - ds_snowpack_hrrrak.HS_mod) * 100,
    'swe_mm': ds_snowpack_hrrrak_lapserate.SWE - ds_snowpack_hrrrak.SWE,
})
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for ax, variable, ylabel in zip(axes, ['snow_depth_cm', 'swe_mm'],
                               ['snow-depth difference (cm)', 'SWE difference (mm)']):
    ax.plot(difference.timestamp, difference[variable], color='0.35', linewidth=2)
    ax.axhline(0, color='black', linewidth=1, linestyle='--')
    ax.set_ylabel(ylabel)
    format_axis(ax)
axes[0].set_xlabel('')
fig.suptitle('TRAM WY2026: Lapse-rate Corrected − Uncorrected HRRR-AK')
fig.tight_layout()
plt.show()


## Summary for the available simulation period

Peak dates use the first timestamp attaining the maximum. End values refer to
the last available model output, which may precede September 30.


In [ ]:
summary_rows = []
for label, ds in models.items():
    depth = ds.HS_mod.to_series() * 100
    swe = ds.SWE.to_series()
    summary_rows.append({
        'Simulation': label,
        'Start (UTC)': depth.index[0],
        'End (UTC)': depth.index[-1],
        'Records': len(depth),
        'Peak snow depth (cm)': depth.max(),
        'Peak depth date (UTC)': depth.idxmax(),
        'Peak SWE (mm)': swe.max(),
        'Peak SWE date (UTC)': swe.idxmax(),
        'Final snow depth (cm)': depth.iloc[-1],
        'Final SWE (mm)': swe.iloc[-1],
    })
summary = pd.DataFrame(summary_rows).set_index('Simulation')
display(summary.round(2))


## January 9, 2026: avalanche-related diagnostics — HRRR-AK forcing

Use **only the lapse-rate corrected TRAM simulation**, labeled **HRRR-AK forcing**.
The reported increase in avalanche danger is supplied as event context, not a
measured danger-rating time series loaded into this notebook. Event days here
use **America/Juneau**, as requested: January 9 is **09:00 UTC January 9 through
09:00 UTC January 10**, with the end excluded for instantaneous diagnostics.
Earlier overview plots retain their original UTC dates.

**Important file distinction:** the supplied August-run `.haz` contains only
July 29–August 1. It is restart memory, not an annual hazard-rating record.
A separate replay through January 10 at 09:00 UTC recovered a January snapshot
in `output/jan09_2026_diagnostics/`. Its 2,428 overlapping hourly records matched
**every stored column of the original simulation exactly**. Only input/output
paths and profile writing were changed; original files were not changed.
The replay configuration, log, outputs, and SHA-256 manifest are archived there.

The `.haz` writer retains 144 half-hour entries, with hoar/drift populated only
for the final 48 entries. These checkpoint buffers should not be treated as
continuous operational danger levels or substituted for the hourly `HN3`/`HN24`
diagnostics. Its header provides no unit metadata, so its raw values below are
explicitly labeled **native stored units**. Use the hourly `.smet` output for
unit-aware snowfall, meteorology, and stability comparisons.

References: [SNOWPACK hazard-file format](https://snowpack.slf.ch/doc-release/html/smet.html),
[hazard checkpoint writer](https://raw.githubusercontent.com/snowpack-model/snowpack/master/Source/snowpack/snowpack/plugins/SmetIO.cc),
[hazard/diagnostic variable definitions](https://snowpack.slf.ch/doc-release/html/structProcessDat.html).


In [ ]:
from zoneinfo import ZoneInfo
import hashlib
import json
import configparser

EVENT_LABEL = 'HRRR-AK forcing'
EVENT_TZ = 'America/Juneau'
EVENT_START = pd.Timestamp('2026-01-09', tz=EVENT_TZ)
EVENT_END = EVENT_START + pd.Timedelta(days=1)
EVENT_WINDOW_START = EVENT_START - pd.Timedelta(days=8)
EVENT_WINDOW_END = EVENT_START + pd.Timedelta(days=4)
EVENT_OUTPUT_DIR = Path('/home/cassie/python/models/run_snowpack/sites/tram/output')
EVENT_PREFIX = 'hrrrak_lapserate_tram_WY2026_base'
EVENT_ARCHIVE = EVENT_OUTPUT_DIR / 'jan09_2026_diagnostics'

print(f'Event day: {EVENT_START} to {EVENT_END} (end exclusive for states)')
print(f'UTC equivalent: {EVENT_START.tz_convert("UTC")} to {EVENT_END.tz_convert("UTC")}')

# Load this run explicitly; the raw HRRR simulation is not used in this section.
event_model, event_header = read_smet(EVENT_OUTPUT_DIR / f'{EVENT_PREFIX}.smet')
assert event_header['station_id'] == 'hrrrak_lapserate_tram_WY2026'
assert float(event_header.get('tz', '0')) == 0


def local_frame(ds):
    frame = ds.to_dataframe()
    frame.index = pd.DatetimeIndex(frame.index).tz_localize('UTC').tz_convert(EVENT_TZ)
    return frame


def event_rows(frame, interval_ending=False):
    if interval_ending:
        # Hourly flux at t describes the preceding interval (t-1h, t].
        return frame.loc[(frame.index > EVENT_START) & (frame.index <= EVENT_END)]
    return frame.loc[(frame.index >= EVENT_START) & (frame.index < EVENT_END)]


In [ ]:
# Audit the supplied file before using any of its data for the event.
haz_original, haz_original_header = read_smet(EVENT_OUTPUT_DIR / f'{EVENT_PREFIX}.haz')
haz_original_frame = local_frame(haz_original)
haz_snapshot, haz_snapshot_header = read_smet(EVENT_ARCHIVE / f'{EVENT_PREFIX}.haz')
haz_snapshot_frame = local_frame(haz_snapshot)

haz_coverage = pd.DataFrame([
    {'File': 'Original August-run .haz', 'Rows': len(haz_original_frame),
     'Start (Alaska)': haz_original_frame.index.min(),
     'End (Alaska)': haz_original_frame.index.max(),
     'Jan 9 interval-ending records': len(event_rows(haz_original_frame, True))},
    {'File': 'Recovered event snapshot .haz', 'Rows': len(haz_snapshot_frame),
     'Start (Alaska)': haz_snapshot_frame.index.min(),
     'End (Alaska)': haz_snapshot_frame.index.max(),
     'Jan 9 interval-ending records': len(event_rows(haz_snapshot_frame, True))},
]).set_index('File')
display(haz_coverage)

manifest = json.loads((EVENT_ARCHIVE / 'manifest.json').read_text())
for source_path, expected_hash in manifest['source_sha256'].items():
    if hashlib.sha256(Path(source_path).read_bytes()).hexdigest() != expected_hash:
        raise ValueError(f'Source changed since the validated replay: {source_path}')
replay_model, _ = read_smet(EVENT_ARCHIVE / f'{EVENT_PREFIX}.smet')
original_overlap = event_model.sel(timestamp=replay_model.timestamp)
if set(replay_model.data_vars) != set(original_overlap.data_vars):
    raise ValueError('Replay variables differ from the original simulation')
for variable in replay_model.data_vars:
    np.testing.assert_array_equal(replay_model[variable], original_overlap[variable])
assert len(event_rows(haz_original_frame, True)) == 0
assert len(event_rows(haz_snapshot_frame, True)) == 48
print(f'Replay verified against {replay_model.sizes["timestamp"]:,} original hourly records.')
print('Original .haz contains no event-day data; subsequent .haz results use the archived replay.')

haz_event_native = event_rows(haz_snapshot_frame, interval_ending=True)
haz_native_summary = haz_event_native.agg(['count', 'min', 'max']).T
haz_native_summary.index.name = 'Checkpoint field (native stored units)'
display(haz_native_summary)


In [ ]:
# Checkpoint buffers only: these are not avalanche danger ratings.
def event_axis(ax):
    ax.axvspan(EVENT_START, EVENT_END, color='red', alpha=0.10)
    ax.axvline(EVENT_START, color='red', linewidth=1.2)
    ax.xaxis.set_major_locator(mdates.DayLocator(tz=ZoneInfo(EVENT_TZ)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b\n%Y', tz=ZoneInfo(EVENT_TZ)))
    ax.set_xlabel('')
    ax.grid(alpha=0.3)

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
for ax, field in zip(axes, haz_snapshot_frame.columns):
    ax.plot(haz_snapshot_frame.index, haz_snapshot_frame[field], color='tab:blue', linewidth=1.8)
    ax.set_ylabel(f'{field}\n(native units)', fontsize=10)
    event_axis(ax)
axes[-1].set_xlim(haz_snapshot_frame.index.min(), haz_snapshot_frame.index.max())
fig.suptitle('HRRR-AK forcing: recovered .haz checkpoint — January 9 shaded (Alaska time)')
fig.tight_layout()
plt.show()


### Retained hourly diagnostics around the event

`HN3` and `HN24` are model new-snow depths over their stated windows;
`HN72_24` is a three-day sum of daily new-snow depths. These are plotted in cm
after applying the SMET unit metadata; they are **not summed over hourly rows**.
`MS_Snow` and `MS_Rain` are hourly mass-flux rates (kg m⁻² h⁻¹, numerically mm
water equivalent per hour). Summing 24 complete, one-hour intervals gives a
daily water-equivalent amount. These output fields reflect modeled snowfall
and rain reaching the snow surface; do not assume they partition the input
precipitation identically at every output hour.

Natural (`Sn`) and skier (`Ss`) stability indices are model diagnostics, not
regional avalanche-danger ratings. We retain their continuous values without
assigning danger classes. The red line marks local midnight January 9; red
shading marks that complete local day.


In [ ]:
event_fields = {
    'HS_mod': 'm', 'HN3': 'm', 'HN24': 'm', 'HN72_24': 'm',
    'TA': 'K', 'VW': 'm/s', 'MS_Snow': 'kg/m2/h', 'MS_Rain': 'kg/m2/h',
    'Sn': '-', 'Ss': '-', 'Sclass2': '-',
}
for field, units in event_fields.items():
    if event_model[field].attrs['units'] != units:
        raise ValueError(f'Unexpected {field} units: {event_model[field].attrs["units"]}')
event_hourly = pd.DataFrame({
    'snow_depth_cm': event_model.HS_mod.values * 100,
    'new_snow_3h_cm': event_model.HN3.values * 100,
    'new_snow_24h_cm': event_model.HN24.values * 100,
    'new_snow_3daily_cm': event_model.HN72_24.values * 100,
    'air_temp_C': event_model.TA.values - 273.15,
    'wind_m_s': event_model.VW.values,
    'snow_mm_h': event_model.MS_Snow.values,
    'rain_mm_h': event_model.MS_Rain.values,
    'Sn': event_model.Sn.values, 'Ss': event_model.Ss.values,
    'model_stability_code': event_model.Sclass2.values,
}, index=pd.DatetimeIndex(event_model.timestamp.values).tz_localize('UTC').tz_convert(EVENT_TZ))
if not (event_hourly.index.to_series().diff().dropna() == pd.Timedelta(hours=1)).all():
    raise ValueError('Hourly flux integration requires uninterrupted one-hour output')
event_window = event_hourly.loc[
    (event_hourly.index >= EVENT_WINDOW_START) & (event_hourly.index <= EVENT_WINDOW_END)
]
if len(event_rows(event_hourly)) != 24:
    raise ValueError('The complete event day is not present in the original output')


In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 12), sharex=True)
for column, style, label in [
    ('new_snow_3h_cm', ':', '3-hour new-snow depth'),
    ('new_snow_24h_cm', '-', '24-hour new-snow depth'),
    ('new_snow_3daily_cm', '--', 'Three daily new-snow depths summed'),
]:
    axes[0].plot(event_window.index, event_window[column], color='tab:blue',
                 linestyle=style, linewidth=2, label=label)
axes[0].set_ylabel('new snow (cm)')
axes[0].legend(loc='upper left', fontsize=9)
axes[1].plot(event_window.index, event_window.wind_m_s, color='tab:blue', linewidth=2)
axes[1].set_ylabel('wind speed (m/s)')
axes[2].plot(event_window.index, event_window.air_temp_C, color='tab:blue', linewidth=2)
axes[2].axhline(0, color='black', linestyle='--', linewidth=1)
axes[2].set_ylabel('air temperature (°C)')
axes[3].plot(event_window.index, event_window.snow_mm_h, color='tab:blue',
             linestyle='--', linewidth=2, label='Solid precipitation rate')
axes[3].plot(event_window.index, event_window.rain_mm_h, color='tab:blue',
             linestyle='-', linewidth=2, label='Rain rate')
axes[3].set_ylabel('water-equivalent\nrate (mm/h)')
axes[3].legend(loc='upper left', fontsize=9)
for ax in axes:
    event_axis(ax)
axes[-1].set_xlim(EVENT_WINDOW_START, EVENT_WINDOW_END)
fig.suptitle('HRRR-AK forcing: loading and meteorology — January 9 shaded (Alaska time)')
fig.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
for ax, field, ylabel in zip(axes[:2], ['Sn', 'Ss'],
                            ['natural stability index Sn', 'skier stability index Ss']):
    ax.plot(event_window.index, event_window[field], color='tab:blue', linewidth=2)
    ax.set_ylabel(ylabel)
# Display the model code without interpreting it as a 1–5 public danger scale.
axes[2].step(event_window.index, event_window.model_stability_code,
             where='post', color='tab:blue', linewidth=2)
axes[2].set_ylabel('model stability code\n(not danger rating)')
axes[2].set_yticks(sorted(event_window.model_stability_code.dropna().unique()))
for ax in axes:
    event_axis(ax)
axes[-1].set_xlim(EVENT_WINDOW_START, EVENT_WINDOW_END)
fig.suptitle('HRRR-AK forcing: model stability diagnostics — January 9 shaded (Alaska time)')
fig.tight_layout()
plt.show()


### Daily context and event-day values

State extrema use **[00:00, next 00:00)** in Alaska time. Precipitation totals
use interval-ending samples **(00:00, next 00:00]**, one hour each; incomplete
days are not assigned a total. Inspect preceding days as well as January 9:
loading and weak-layer evolution may precede the reported danger increase.


In [ ]:
daily_rows = []
for day in pd.date_range(EVENT_START - pd.Timedelta(days=7),
                         EVENT_START + pd.Timedelta(days=3), freq='D'):
    end = day + pd.Timedelta(days=1)
    states = event_hourly.loc[(event_hourly.index >= day) & (event_hourly.index < end)]
    fluxes = event_hourly.loc[(event_hourly.index > day) & (event_hourly.index <= end)]
    daily_rows.append({
        'Local date': day.strftime('%Y-%m-%d'),
        'Event day': day == EVENT_START,
        'Hours': len(states),
        'Max HN24 (cm)': states.new_snow_24h_cm.max(),
        'Max 3-daily HN (cm)': states.new_snow_3daily_cm.max(),
        'Min TA (°C)': states.air_temp_C.min(),
        'Max TA (°C)': states.air_temp_C.max(),
        'Max wind (m/s)': states.wind_m_s.max(),
        'Min Sn': states.Sn.min(), 'Min Ss': states.Ss.min(),
        'Solid precip (mm)': fluxes.snow_mm_h.sum(min_count=24) if len(fluxes) == 24 else np.nan,
        'Rain (mm)': fluxes.rain_mm_h.sum(min_count=24) if len(fluxes) == 24 else np.nan,
    })
daily_event_summary = pd.DataFrame(daily_rows).set_index('Local date')
display(daily_event_summary.round(3))
print('January 9 model state values (Alaska time):')
display(event_rows(event_hourly).round(3))


In [ ]:
event_states = event_rows(event_hourly)
event_fluxes = event_rows(event_hourly, interval_ending=True)
print('HRRR-AK forcing — January 9, 2026 (Alaska local day)')
print(f'24-hour new-snow depth maximum: {event_states.new_snow_24h_cm.max():.2f} cm')
print(f'Air temperature range: {event_states.air_temp_C.min():.2f} to {event_states.air_temp_C.max():.2f} °C')
print(f'Maximum wind speed: {event_states.wind_m_s.max():.2f} m/s')
print(f'Rain reaching the snow surface: {event_fluxes.rain_mm_h.sum(min_count=24):.2f} mm')
print(f'Natural stability index range: {event_states.Sn.min():.3f} to {event_states.Sn.max():.3f}')
print(f'Skier stability index range: {event_states.Ss.min():.3f} to {event_states.Ss.max():.3f}')


### Initial interpretation

The model contains signals consistent with changing instability conditions around
the reported event: substantial snowfall on January 5–6 is followed by warming,
rain, and stronger winds on January 9. For the local January 9 day, the retained
output gives **15.49 cm maximum HN24**, temperature **−2.06 to +3.79°C**, maximum
wind **11.74 m/s**, and **33.05 mm** of modeled rain reaching the snow surface.
The natural stability index spans **0.893–2.899** and the skier index
**0.229–1.040**. The skier-index minimum is lower than the previous local day's
0.990; the largest recent new-snow loading and natural-index reduction occurred
earlier, so January 9 is not a unique peak in every diagnostic.

These are **supporting model signals**, not a validated prediction of the reported
increase in avalanche danger. The indices are empirical snowpack diagnostics;
strength/stress indices generally indicate greater susceptibility at lower
values, but a day-level minimum alone does not establish avalanche occurrence,
problem type, or regional danger. In particular, do not interpret the model's
numeric stability code as a public 1–5 danger rating, or zero/missing checkpoint
drift as evidence that wind slabs were absent. Inspect the hourly timing and
local layers before deciding whether the model captured the relevant process.

This is one point/slope simulation at 529 m, with a 38° north-facing setup.
A defensible event evaluation still needs the contemporaneous danger bulletin or
avalanche record (timing, elevation/aspect, and problem type) and ideally profile/
weak-layer evidence from the `.pro` file. No observed danger-rating series or
avalanche occurrence labels have been loaded here. No calibrated warning
threshold or binary hit/miss verdict is assigned.

Sources for interpretation: [SNOWPACK stability methods](https://snowpack.slf.ch/doc-dev/html/classStability.html),
[Reuter et al., snow-instability indicators and their evaluation](https://doi.org/10.1016/j.coldregions.2021.103462).
